# WarLens — Model Evaluation

This notebook produces rigorous evaluation metrics for the ResNet50 and MobileNetV2 models:
- Confusion matrix
- Per-class precision, recall, F1
- t-SNE feature space visualization
- Model comparison bar chart

All output plots are saved to `../figures/`.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.manifold import TSNE
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess

DATASET_DIR = '../dataset'
FIGURES_DIR = '../figures'
MODEL_RESNET_PATH = 'war_lens_resnet50.h5'
MODEL_MOBILENET_PATH = 'war_lens_mobilenetv2.h5'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
RANDOM_SEED = 42

CLASS_NAMES = ['Combat', 'Destroyed Buildings', 'Fire', 'Humanitarian Aid', 'Military Vehicles & Weapons']

os.makedirs(FIGURES_DIR, exist_ok=True)
print('TensorFlow', tf.__version__)

## 1. Load Dataset

In [ ]:
# Use 20% of the dataset as a held-out test split
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

test_generator = datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=RANDOM_SEED
)

print('Classes:', test_generator.class_indices)
print('Test samples:', test_generator.samples)

## 2. Load Models

In [ ]:
model_resnet = None
model_mobilenet = None

if os.path.exists(MODEL_RESNET_PATH):
    model_resnet = load_model(MODEL_RESNET_PATH)
    print('ResNet50 loaded')
else:
    print(f'ResNet50 model not found at {MODEL_RESNET_PATH}')

if os.path.exists(MODEL_MOBILENET_PATH):
    model_mobilenet = load_model(MODEL_MOBILENET_PATH)
    print('MobileNetV2 loaded')
else:
    print(f'MobileNetV2 model not found at {MODEL_MOBILENET_PATH}')

## 3. Run Predictions

In [ ]:
def get_predictions(model, generator, preprocess_fn):
    generator.reset()
    # Re-apply model-specific preprocessing on top of the rescaled images
    all_preds = []
    all_labels = []
    for i in range(len(generator)):
        x, y = generator[i]
        x_processed = preprocess_fn(x * 255.0)  # undo rescale then reapply proper preprocessing
        preds = model.predict(x_processed, verbose=0)
        all_preds.append(preds)
        all_labels.append(y)
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    return all_preds, np.argmax(all_labels, axis=1)

true_labels = None
preds_resnet = preds_mobilenet = preds_ensemble = None

if model_resnet:
    preds_resnet, true_labels = get_predictions(model_resnet, test_generator, resnet_preprocess)
    print('ResNet50 predictions done')

if model_mobilenet:
    preds_mobilenet, true_labels = get_predictions(model_mobilenet, test_generator, mobilenet_preprocess)
    print('MobileNetV2 predictions done')

if preds_resnet is not None and preds_mobilenet is not None:
    preds_ensemble = (preds_resnet + preds_mobilenet) / 2.0
    print('Ensemble predictions computed')

## 4. Confusion Matrix

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title, filename):
    cm = confusion_matrix(y_true, y_pred)
    cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        cm_normalized, annot=True, fmt='.2f', cmap='Blues',
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        linewidths=0.5, ax=ax
    )
    ax.set_title(title, fontsize=14, pad=12)
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('True', fontsize=11)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    path = os.path.join(FIGURES_DIR, filename)
    plt.savefig(path, dpi=150)
    plt.show()
    print(f'Saved to {path}')

if preds_resnet is not None:
    plot_confusion_matrix(
        true_labels, np.argmax(preds_resnet, axis=1),
        'Confusion Matrix — ResNet50', 'confusion_matrix_resnet50.png'
    )

if preds_mobilenet is not None:
    plot_confusion_matrix(
        true_labels, np.argmax(preds_mobilenet, axis=1),
        'Confusion Matrix — MobileNetV2', 'confusion_matrix_mobilenetv2.png'
    )

if preds_ensemble is not None:
    plot_confusion_matrix(
        true_labels, np.argmax(preds_ensemble, axis=1),
        'Confusion Matrix — Ensemble', 'confusion_matrix_ensemble.png'
    )

## 5. Per-Class Metrics (Precision, Recall, F1)

In [ ]:
for name, preds in [('ResNet50', preds_resnet), ('MobileNetV2', preds_mobilenet), ('Ensemble', preds_ensemble)]:
    if preds is None:
        continue
    print(f'\n=== {name} ===')
    print(classification_report(
        true_labels, np.argmax(preds, axis=1),
        target_names=CLASS_NAMES
    ))

## 6. Model Comparison Bar Chart

In [ ]:
from sklearn.metrics import accuracy_score

results = {}
if preds_resnet is not None:
    results['ResNet50'] = accuracy_score(true_labels, np.argmax(preds_resnet, axis=1)) * 100
if preds_mobilenet is not None:
    results['MobileNetV2'] = accuracy_score(true_labels, np.argmax(preds_mobilenet, axis=1)) * 100
if preds_ensemble is not None:
    results['Ensemble'] = accuracy_score(true_labels, np.argmax(preds_ensemble, axis=1)) * 100

if results:
    fig, ax = plt.subplots(figsize=(7, 4))
    colors = ['#2563eb', '#16a34a', '#9333ea']
    bars = ax.bar(list(results.keys()), list(results.values()), color=colors[:len(results)], width=0.4)
    ax.set_ylim(80, 100)
    ax.set_ylabel('Accuracy (%)', fontsize=11)
    ax.set_title('Model Accuracy Comparison', fontsize=13)
    for bar, val in zip(bars, results.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=10)
    plt.tight_layout()
    path = os.path.join(FIGURES_DIR, 'model_comparison.png')
    plt.savefig(path, dpi=150)
    plt.show()
    print(f'Saved to {path}')

## 7. t-SNE Feature Space Visualization

In [ ]:
if model_resnet is not None:
    # Extract penultimate layer (before the final Dense) activations
    feature_model = Model(
        inputs=model_resnet.input,
        outputs=model_resnet.layers[-2].output  # GlobalAveragePooling2D output
    )

    test_generator.reset()
    features = []
    labels_for_tsne = []
    for i in range(len(test_generator)):
        x, y = test_generator[i]
        x_processed = resnet_preprocess(x * 255.0)
        feat = feature_model.predict(x_processed, verbose=0)
        features.append(feat)
        labels_for_tsne.extend(np.argmax(y, axis=1))

    features = np.vstack(features)
    labels_for_tsne = np.array(labels_for_tsne)

    print(f'Running t-SNE on {features.shape[0]} samples with {features.shape[1]} dimensions...')
    tsne = TSNE(n_components=2, random_state=RANDOM_SEED, perplexity=30, n_iter=1000)
    embeddings = tsne.fit_transform(features)

    colors_tsne = ['#ef4444', '#f97316', '#eab308', '#22c55e', '#3b82f6']
    fig, ax = plt.subplots(figsize=(9, 7))
    for class_idx, (class_name, color) in enumerate(zip(CLASS_NAMES, colors_tsne)):
        mask = labels_for_tsne == class_idx
        ax.scatter(
            embeddings[mask, 0], embeddings[mask, 1],
            c=color, label=class_name, alpha=0.7, s=40, edgecolors='none'
        )
    ax.legend(loc='best', framealpha=0.9)
    ax.set_title('t-SNE of ResNet50 Feature Space', fontsize=13)
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    plt.tight_layout()
    path = os.path.join(FIGURES_DIR, 'tsne_features.png')
    plt.savefig(path, dpi=150)
    plt.show()
    print(f'Saved to {path}')